# CS310 Natural Language Processing
# Lab 2: Neural Text Classification

In this lab, we will practice building a neural network for text classification. 

The tutorial code is adopted from the official PyTorch tutorial: *Text classification with the torchtext library*

https://pytorch.org/tutorials/beginner/text_sentiment_ngrams_tutorial.html#text-classification-with-the-torchtext-library

## Install datasets

Url: https://huggingface.co/docs/datasets/en/installation
```bash
conda install -c huggingface -c conda-forge datasets
```

In [ ]:
# 导入 PyTorch 以及后面要用到的数据切分工具
import torch
from torch.utils.data.dataset import random_split

# 从 data_utils.py 导入实验里自定义的数据处理组件
# DatasetIterator: 把 Hugging Face 数据集包装成 (sentence, label) 迭代器
# get_tokenizer: 返回分词函数
# build_vocab_from_iter: 根据训练语料构建词表
# to_map_style_dataset: 把可迭代数据集转成可按索引访问的数据集
from data_utils import DatasetIterator, get_tokenizer, build_vocab_from_iter, to_map_style_dataset

In [ ]:
# 导入 Hugging Face 的数据集加载函数
from datasets import load_dataset  # 首次下载需要联网，速度可能较慢

# 加载 GLUE 基准中的 SST-2 情感分类数据集
# 返回结果 ds 类似一个字典，里面包含 train、validation、test 等切分
ds = load_dataset('glue', 'sst2')  # 这一步可能需要等待一段时间

In [ ]:
# 先查看原始数据长什么样，确认每条样本的格式是否符合预期
train_iter = DatasetIterator(ds['train'])

count = 0
for item in train_iter:
    print(item)  # 每条 item 是 (sentence, label)
    count += 1
    if count > 5:
        break  # 只看前几条，避免输出过多

## Apply Tokenizer

In [ ]:
# basic_english 会先做简单归一化，再按空格切分
tokenizer = get_tokenizer("basic_english")

def yield_tokens(data_iter):
    # 逐条读取文本，并把句子转换成 token 列表
    for text, _ in data_iter:
        yield tokenizer(text)

In [ ]:
# 检查 yield_tokens() 的输出，确认分词后的结果是否合理
count = 0
for tokens in yield_tokens(DatasetIterator(ds['train'])):  # 这里重新创建迭代器，避免前面的迭代已经耗尽
    print(tokens)
    count += 1
    if count > 3:
        break

## Build Vocabulary

First, we build the vocabulary using the `build_vocab_from_iterator` function.

In [ ]:
# 用训练集的 token 序列构建词表，并把未知词 <unk> 放进特殊符号集合里
vocab = build_vocab_from_iter(yield_tokens(DatasetIterator(ds['train'])), specials=["<unk>"])

You can see that all strings are converted into integer IDs.

In [ ]:
# 检查词表如何把单词映射成整数 id
print(vocab(['here', 'is', 'an', 'example']))
print(vocab(['hide', 'new', 'secretions', 'from', 'the', 'parental', 'units']))
print(vocab(['of', 'saucy']))

# 未登录词（训练词表里没见过的词）会被映射到 <unk> 对应的 id
print(vocab(['here', 'is', 'a', '@#$@!#$%']))

Next, further define the `text_pipeline` and `label_pipeline` functions, for converting strings to integers. 

In [ ]:
# text_pipeline: 原始句子 -> 分词 -> 词 id 序列
text_pipeline = lambda x: vocab(tokenizer(x))

# label_pipeline: 标签 -> 整数类别 id
label_pipeline = lambda x: int(x)

# 测试 text_pipeline()
tokens = text_pipeline('here is an example')
print(tokens)

# 测试 label_pipeline()
lbl = label_pipeline('1')
print(lbl)

## T1. Batchify Data 

Your goal is to define the `Collate_batch` function, which will be used to process the "raw" data batch.

*Hint*: In the loop of `collate_batch`, the labels, tokens, and the offsets of all the examples are collected into three lists. Finally, the lists are converted into tensors. 

*Hint*: The `offsets` need to contain the cumulative positions of tokens in the batch. 
For example, if the batch contains 3 examples, whose lengths are `[1,3,2]`, then the final offsets should be `[0,1,4]`.

In [ ]:
from torch.utils.data import DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def collate_batch(batch):
    # 分别收集标签、每条样本的 token id，以及每条样本在拼接长向量中的起始位置
    label_list, token_ids_list, offsets = [], [], [0]
    for _text, _label in batch:
        # 把标签转成整数
        label_list.append(label_pipeline(_label))

        # 把一句话转成 token id 张量
        token_ids = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        token_ids_list.append(token_ids)

        # 先记录每条样本的长度，后面再通过累积和转成 offsets
        offsets.append(token_ids.size(0))

    ### START YOUR CODE ### (complete the code to create labels, token_ids, and offsets)
    # 所有标签拼成一个一维张量
    labels = torch.tensor(label_list, dtype=torch.int64)

    # offsets 需要表示每条句子的起始下标，所以对长度做累积和
    offsets = torch.tensor(offsets[:-1], dtype=torch.int64).cumsum(dim=0)

    # 把一个 batch 里所有句子的 token id 首尾拼接成一个长向量
    token_ids = torch.cat(token_ids_list)
    ### END YOUR CODE ###

    # 把张量移动到 CPU 或 GPU
    return labels.to(device), token_ids.to(device), offsets.to(device)

Next, use the defined `collate_batch` function to create the dataloader.

In [ ]:
# 用 collate_batch 告诉 DataLoader 如何把变长文本组织成一个 batch
train_iter = DatasetIterator(ds['train'])

dataloader = DataLoader(
    train_iter, batch_size=8, shuffle=False, collate_fn=collate_batch
)

In [ ]:
# 从 dataloader 里取出一个 batch，检查 labels、token_ids、offsets 的含义
for i, (labels, token_ids, offsets) in enumerate(dataloader):
    if i == 0:
        break

# offsets 记录的是每个样本在 token_ids 长向量中的起始位置
print('Number of tokens in this batch: ', token_ids.size(0))
print('Number of examples in one batch: ', labels.size(0))
print('Example 0: ', token_ids[offsets[0]:offsets[1]])
print('Example 7: ', token_ids[offsets[7]:])

# 你应该看到类似下面的输出：
# Number of tokens in this batch:  82
# Number of examples in one batch:  8
# Example 0:  tensor([ 4460,    92, 12405,    38,     1,  7259,  8990])
# Example 7:  tensor([   5, 6549])

## T2. Define the Model

The model consists of two parts of parameters: an embedding layer and a fully connected layer.

Your need to first initialize an `nn.EmbeddingBag` instance and a `nn.Linear` instance:
- The embedding layer should be initialized with `vocab_size`, `embed_dim`, and `sparse=False`.
- The fully connected layer should have `embed_dim` as input size and `num_class` as output size.

Then, in the `forward` function, the embedding layer should called with `token_ids` and `offsets` as inputs. The output of embedding layer is fed to the fully connected layer to get the final output. 

In [ ]:
from torch import nn

class TextClassificationModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(TextClassificationModel, self).__init__()
        ### START YOUR CODE ###
        # EmbeddingBag 会把每条句子的多个 token 向量聚合成一个固定长度的表示
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False)

        # 全连接层把句向量映射到类别空间
        self.fc = nn.Linear(embed_dim, num_class)
        ### END YOUR CODE ###
        self.init_weights()

    def init_weights(self):
        # 用较小的随机数初始化参数，避免一开始数值过大
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, token_ids, offsets):
        ### START YOUR CODE ###
        # 根据 token_ids 和 offsets 得到每条句子的 embedding 表示
        embedded = self.embedding(token_ids, offsets)

        # 输出每个类别的分数（logits）
        out = self.fc(embedded)
        ### END YOUR CODE ###
        return out

In [ ]:
# 构建模型所需的几个关键参数
train_iter = DatasetIterator(ds['train'])
num_class = len(set([label for (_, label) in train_iter]))  # 类别数
vocab_size = len(vocab)  # 词表大小
emsize = 64  # 每个词向量的维度
model = TextClassificationModel(vocab_size, emsize, num_class).to(device)

In [ ]:
# 先做一次前向传播，检查模型输出的维度是否正确
model.eval()  # 切换到评估模式
with torch.no_grad():
    for i, (labels, token_ids, offsets) in enumerate(dataloader):
        output = model(token_ids, offsets)
        if i == 0:
            break

print('output size:', output.size())

# 你应该看到如下形状：
# output size: torch.Size([8, 2])

## T3. Define the loss function

Cross entropy loss should be used. You can use `torch.nn.CrossEntropyLoss` to define the loss function.

In [ ]:
# 训练超参数
EPOCHS = 10  # 训练轮数
LR = 5  # 学习率
BATCH_SIZE = 8  # 每个 batch 中的样本数

### START YOUR CODE ###
# 交叉熵损失适用于多分类任务，这里是二分类
criterion = torch.nn.CrossEntropyLoss()
### END YOUR CODE ###

# 使用 SGD 更新模型参数
optimizer = torch.optim.SGD(model.parameters(), lr=LR)

# 每个 epoch 后可以按策略衰减学习率
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.1)

Test if the loss function `criterion` works

In the cell below, implement the **manual computation** of cross entropy loss. Use the formula $-y_i\log(\hat{y_i})$, 

where $y_i$ is the $i$-th ground truth label in `labels`, and $\hat{y_i}$ is the predicted probability in `output` of the correponding label.

In [ ]:
# 先取一个 batch 的输出和标签，用来验证损失函数是否工作正常
model.eval()
with torch.no_grad():
    for i, (labels, token_ids, offsets) in enumerate(dataloader):
        output = model(token_ids, offsets)
        # print(f"batch {i} output: {output}")
        if i == 0:
            break
print('output shape:', output.shape)

# 使用 PyTorch 内置交叉熵损失
loss = criterion(output, labels)
print('loss:', loss)

# 手动计算交叉熵，帮助理解其含义
loss_manual = []
for i in range(output.shape[0]):
    ### START YOUR CODE ###
    # 先把当前样本的 logits 转成概率分布
    probs = torch.softmax(output[i], dim=0)

    # 取出真实标签对应的概率，并计算 -log(p)
    l = -torch.log(probs[labels[i]])
    ### END YOUR CODE ###
    loss_manual.append(l)
loss_manual = torch.stack(loss_manual)
print('loss_manual mean:', loss_manual.mean())
print('loss equals loss_manual:', torch.isclose(loss, loss_manual.mean()))

# 你应该看到如下输出：
# output shape: torch.Size([8, 2])
# loss: tensor(XXXXX)
# loss_manual mean: tensor(XXXXX)
# loss equals loss_manual: tensor(True)

## T4. Train and Evaluate Functions

Define train and evaluate functions.

You need to implement the forward pass, loss computation, backward propagation, and parameter update in the `train` function. 

Also, for each batch of data, calculate the total number of correctly predicted examples, by comparing `output` and `labels`.

In [ ]:
import time

def train(model, dataloader, optimizer, criterion, epoch: int):
    model.train()
    total_acc, total_count = 0, 0
    log_interval = 500
    start_time = time.time()

    for idx, (labels, token_ids, offsets) in enumerate(dataloader):
        # 清空上一个 batch 的梯度
        optimizer.zero_grad()
        ### START YOUR CODE ###
        # 前向传播：输入当前 batch，得到类别分数
        output = model(token_ids, offsets)
        ### END YOUR CODE ###
        try:
            ### START YOUR CODE ###
            # 计算当前 batch 的损失
            loss = criterion(output, labels)
            ### END YOUR CODE ###
        except Exception:
            print('Error in loss calculation')
            print('output: ', output.size())
            print('labels: ', labels.size())
            print('token_ids: ', token_ids)
            print('offsets: ', offsets)
            raise
        ### START YOUR CODE ###
        # 反向传播计算梯度
        loss.backward()

        # 做梯度裁剪，避免梯度过大导致训练不稳定
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)

        # 根据梯度更新参数
        optimizer.step()
        ### END YOUR CODE ###

        ### START YOUR CODE ###
        # 统计当前 batch 预测正确的样本数
        total_acc += (output.argmax(1) == labels).sum().item()
        ### END YOUR CODE ###

        total_count += labels.size(0)
        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time() - start_time
            print(
                "| epoch {:3d} | {:5d}/{:5d} batches "
                "| accuracy {:8.3f}".format(
                    epoch, idx, len(dataloader), total_acc / total_count
                )
            )
            total_acc, total_count = 0, 0
            start_time = time.time()

@torch.no_grad()
def evaluate(model, dataloader, criterion):
    model.eval()
    total_acc, total_count = 0, 0

    for idx, (label, text, offsets) in enumerate(dataloader):
        ### START YOUR CODE ###
        # 评估阶段仍然要前向传播和算损失，但不做反向传播与参数更新
        output = model(text, offsets)
        loss = criterion(output, label)
        total_acc += (output.argmax(1) == label).sum().item()
        ### END YOUR CODE ###
        total_count += label.size(0)

    return total_acc / total_count

## Load train, valid, and test data

In [ ]:
# 准备 train、valid、test 数据
train_iter = DatasetIterator(ds['train'])
test_iter = DatasetIterator(ds['test'])
train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)

# 从训练集里再划出一小部分作为验证集
num_train = int(len(train_dataset) * 0.95)
split_train_, split_valid_ = random_split(
    train_dataset, [num_train, len(train_dataset) - num_train]
)

# 为训练、验证、测试分别构建 dataloader
train_dataloader = DataLoader(
    split_train_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)
valid_dataloader = DataLoader(
    split_valid_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)
test_dataloader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)

## Main Training Loop

In [ ]:
# 主训练循环：每个 epoch 训练一次，再在验证集上评估一次
total_accu = None
for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()

    train(model, train_dataloader, optimizer, criterion, epoch)
    accu_val = evaluate(model, valid_dataloader, criterion)

    # 如果验证集效果变差，就降低学习率
    if total_accu is not None and total_accu > accu_val:
        scheduler.step()
    else:
        total_accu = accu_val

    print("-" * 59)
    print(
        "| end of epoch {:3d} | time: {:5.2f}s | "
        "valid accuracy {:8.3f} ".format(
            epoch, time.time() - epoch_start_time, accu_val
        )
    )
    print("-" * 59)

In [ ]:
# 保存训练好的模型参数，便于后续直接加载使用
torch.save(model.state_dict(), "text_classification_model.pth")

## Evaluate with Test Data

This is a necessary step. But since the `test` split of SST2 is not annotated, we will use the `dev` split here to pretend it is the test data.

In [ ]:
# 这里用验证集近似充当测试集，计算最终准确率
accu_test = evaluate(model, valid_dataloader, criterion)
print("test accuracy {:8.3f}".format(accu_test))

# 你的测试准确率通常应该在 0.9 左右

## Predict

Test the model with a few unannotated examples.

In [ ]:
sentiment_labels = ['negative', 'positive']

def predict(text, model, vocab, tokenizer, labels):
    model.eval()
    with torch.no_grad():
        # 把输入句子转换成 token id 张量
        text = torch.tensor(vocab(tokenizer(text)), device=device)

        # 单句预测时 offsets 只有一个起点 0
        output = model(text, torch.tensor([0], device=device))

        # 取分数最高的类别作为预测结果
        return labels[output.argmax(1).item()]

ex_text_str = "A very well-made, funny and entertaining picture."
print("This is a %s sentiment." % (predict(ex_text_str, model, vocab, tokenizer, sentiment_labels)))

Congratulations! You have successfully built and trained a neural network model to classify sentiment in text data.